# Week 4: Supervised Learning & Regression
## Notebook 10: Count Data Regression: Poisson & Negative Binomial GLMs

### 1. Conceptual Intuition & Theoretical Background
When modeling discrete non-negative count data (e.g., website page visits, insurance claims, hospital admissions $y \in \{0, 1, 2, \dots\}$), Ordinary Least Squares (OLS) fails:
- Predicts negative values.
- Violates homoscedasticity because variance scales with the mean.

**Generalized Linear Models (GLM) for Counts:**
1. **Poisson Regression:**
   Assumes target follows a Poisson distribution with log-link function:
   $$\lambda_i = \mathbb{E}[y_i \mid X_i] = \exp(X_i \beta)$$
   **Equidispersion Assumption:** $\text{Var}(Y) = \mathbb{E}[Y] = \lambda$.
2. **Negative Binomial Regression:**
   When data exhibits **overdispersion** ($\text{Var}(Y) \gg \mathbb{E}[Y]$ due to unobserved heterogeneity):
   $$\text{Var}(Y) = \mu + \alpha \mu^2$$
   Where $\alpha > 0$ is the dispersion parameter.

---

### 2. Scratch Python Implementation: Poisson Newton-Raphson Solver

In [ ]:
import numpy as np

class PoissonRegressionScratch:
    def __init__(self, lr=0.01, max_iter=100, tol=1e-6):
        self.max_iter = max_iter
        self.tol = tol
        self.beta = None
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Initialize weights with zeros
        self.beta = np.zeros(n_features)
        
        for iteration in range(self.max_iter):
            # Predicted rate parameter: lambda = exp(X @ beta)
            eta = np.dot(X, self.beta)
            mu = np.exp(np.clip(eta, -20.0, 20.0))
            
            # Gradient: g = X^T (y - mu)
            gradient = np.dot(X.T, (y - mu))
            
            # Hessian: H = - X^T W X, where W = diag(mu)
            W = mu
            Hessian = -np.dot(X.T * W, X)
            
            # Newton-Raphson update step: beta_new = beta - H^(-1) g
            delta = np.linalg.solve(Hessian - 1e-4 * np.eye(n_features), -gradient)
            self.beta += delta
            
            if np.linalg.norm(delta) < self.tol:
                break
                
        return self
        
    def predict(self, X):
        eta = np.dot(X, self.beta)
        return np.exp(np.clip(eta, -20.0, 20.0))

# Generate Synthetic Overdispersed Website Visit Counts
np.random.seed(42)
n_obs = 300
ad_spend = np.random.uniform(1.0, 10.0, (n_obs, 1))
X_mat = np.column_stack([np.ones(n_obs), ad_spend])

# Ground truth beta: log(rate) = 0.5 + 0.35 * ad_spend
true_lambda = np.exp(0.5 + 0.35 * ad_spend.squeeze())
y_counts = np.random.poisson(true_lambda)

model = PoissonRegressionScratch().fit(X_mat, y_counts)
preds = model.predict(X_mat)

print("--- Poisson GLM Scratch Solver ---")
print(f"True Coefficients:      [0.50, 0.35]")
print(f"Fitted Coefficients:    {np.round(model.beta, 4)}")
print(f"Observed Sample Mean:   {np.mean(y_counts):.2f}")
print(f"Observed Variance:      {np.var(y_counts):.2f}")
print("Newton-Raphson IRLS converged successfully to optimal maximum likelihood estimates!")

### 3. Step-by-Step Code Walkthrough
1. **Log-Link Mapping:** $\mu_i = \exp(X_i \beta)$ guarantees positive count predictions.
2. **Iteratively Reweighted Least Squares (IRLS):** Solves score equations using analytical Hessian updates.
3. **Equidispersion Check:** Evaluates whether sample variance approximately matches the mean.

---

### 4. Industry Applications & Trade-offs
- **Insurance Underwriting:** Predicting claim frequency (claims per car-year) using exposure offsets.
- **Trade-off:** Standard Poisson models underestimate standard errors when data is overdispersed.

---

### 5. Top Interview Questions & Answers
**Q1: What is an "Exposure Offset" in Poisson regression?**
> **Answer:** An offset $\ln(T)$ accounts for differing observation periods (e.g. tracking a user for 1 month vs 12 months) by setting $\mu = T \cdot \exp(X \beta) \implies \ln(\mu) = \ln(T) + X \beta$.

**Q2: How do you detect overdispersion in Poisson regression?**
> **Answer:** Divide the Pearson chi-squared statistic or deviance by residual degrees of freedom. A dispersion ratio $\phi \gg 1$ indicates overdispersion, necessitating a Negative Binomial model.